# 01d - Sweep de Pré-processamento (validação cross-generator)

Varredura estruturada de 5 métodos de pré-processamento. Para cada combinação **(método, valor, seed)**:

1. Aplica o pré-processamento on-the-fly (sem salvar em disco)
2. Treina uma ResNet-50 (10 epochs, 5% do 140k)
3. Avalia a **AUC no ArtiFact** (cross-generator)

A métrica é **AUC**, não acurácia: é threshold-independent e robusta ao desvio de calibração que surge ao mudar de gerador. O conjunto de avaliação do ArtiFact é **fixo e balanceado** (mesmas imagens em todos os runs), de modo que a variância entre seeds reflita só o treino, não o sorteio do teste.

Sweep estruturado com seeds repetidos → cada ponto tem média e desvio, permitindo distinguir sinal de ruído.

| Método | Mecanismo |
|---|---|
| `jpeg` | low-pass por blocos DCT + quantização |
| `blur` | low-pass linear suave |
| `downscale` | low-pass por reamostragem |
| `noise` | mascaramento aditivo |
| `median` | suavização não-linear preserva-bordas |

> Requer o ArtiFact preparado pelo notebook `01_artifact_preparacao.ipynb`.

In [ ]:
import io
import json
import time
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
from torch.utils.data import DataLoader, Subset, Dataset
from torchvision import datasets, models, transforms
from sklearn.metrics import accuracy_score, roc_auc_score
from tqdm.notebook import tqdm   # versão widget: barra fixa em cima, logs rolam embaixo

In [ ]:
PROJECT_ROOT    = Path.cwd().resolve().parent
_data_root_file = PROJECT_ROOT / "data_root.env"
DATA_ROOT       = Path(_data_root_file.read_text().strip()) if _data_root_file.exists() else PROJECT_ROOT / "data"

RAW_DIR        = DATA_ROOT / "raw" / "140k_faces" / "real_vs_fake" / "real-vs-fake"
ARTIFACT_DIR   = DATA_ROOT / "raw" / "artifact_faces"
RESULTS_DIR    = PROJECT_ROOT / "artifacts" / "sweep"
FIGURES_DIR    = PROJECT_ROOT / "reports" / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH   = RESULTS_DIR / "sweep_results.json"

IMAGE_SIZE      = 224
BATCH_SIZE      = 32
NUM_WORKERS     = 0          # 0 no Windows: workers>0 com Dataset/transform customizado deadlocka
NUM_EPOCHS      = 10
LEARNING_RATE   = 1e-4
SAMPLE_FRACTION = 0.05      # do 140k para treino

# avaliação cross-generator: conjunto FIXO e BALANCEADO
ARTIFACT_N_PER_CLASS = 1000   # 1000 real + 1000 fake, iguais em todos os runs
ARTIFACT_EVAL_SEED   = 42     # seed fixo do sorteio do conjunto de teste

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── grade do sweep ────────────────────────────────────────────
# 6 valores por método (pontas + meio) — cobre a curva e corta ~35% do tempo.
# A fase 2 (random search) refina a região vencedora de qualquer jeito.
VALUE_GRID = {
    "jpeg":      [30, 40, 50, 60, 70, 80],
    "blur":      [0.5, 1.0, 1.5, 2.0, 2.5, 3.5],
    "downscale": [1.25, 1.75, 2.25, 3.0, 4.0, 5.0],
    "noise":     [0.01, 0.03, 0.05, 0.09, 0.13, 0.20],
    "median":    [3, 5, 7, 9, 11],
}
N_SEEDS = {"jpeg": 2, "blur": 2, "downscale": 2, "noise": 2, "median": 2}
SEEDS = [42, 123, 7, 2024]
# total = 6*2 + 6*2 + 6*2 + 6*2 + 5*2 = 58
# ──────────────────────────────────────────────────────────────

total_runs = sum(len(VALUE_GRID[m]) * N_SEEDS[m] for m in VALUE_GRID)
print("Device:", DEVICE)
print("Total de runs:", total_runs)
print("ArtiFact:", ARTIFACT_DIR)

## 1. Pré-processamento como Transform

In [ ]:
def apply_preprocess(img, method, value):
    if method == "jpeg":
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=int(value))
        buf.seek(0)
        return Image.open(buf).copy()
    elif method == "blur":
        return img.filter(ImageFilter.GaussianBlur(radius=float(value)))
    elif method == "downscale":
        w, h = img.size
        nw, nh = max(1, int(w / value)), max(1, int(h / value))
        return img.resize((nw, nh), Image.BILINEAR).resize((w, h), Image.BILINEAR)
    elif method == "noise":
        arr = np.array(img, dtype=np.float32)
        arr = arr + np.random.normal(0, float(value) * 255, arr.shape)
        return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
    elif method == "median":
        return img.filter(ImageFilter.MedianFilter(size=int(value)))
    else:
        raise ValueError(method)

class Preprocess:
    def __init__(self, method, value):
        self.method, self.value = method, value
    def __call__(self, img):
        return apply_preprocess(img, self.method, self.value)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def make_transforms(method, value, train=True):
    steps = [Preprocess(method, value), transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))]
    if train:
        steps += [transforms.RandomHorizontalFlip()]
    steps += [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    return transforms.Compose(steps)

print("Transforms definidos.")

## 2. Loaders

In [ ]:
IMG_EXTS = ("*.jpg", "*.jpeg", "*.png", "*.webp")

def list_images(folder):
    files = []
    for e in IMG_EXTS:
        files += list(folder.glob(e))
    return sorted(files)

class PathListDataset(Dataset):
    """Dataset sobre uma lista fixa de (caminho, label), com transform configurável."""
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, label = self.items[i]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label

# ── conjunto de avaliação ArtiFact: FIXO e BALANCEADO (construído uma vez) ──
# mapeamento de classes consistente com ImageFolder do 140k: fake=0, real=1
_rng = random.Random(ARTIFACT_EVAL_SEED)
_real = list_images(ARTIFACT_DIR / "real")
_fake = list_images(ARTIFACT_DIR / "fake")
_real = _rng.sample(_real, min(ARTIFACT_N_PER_CLASS, len(_real)))
_fake = _rng.sample(_fake, min(ARTIFACT_N_PER_CLASS, len(_fake)))
ARTIFACT_ITEMS = [(p, 1) for p in _real] + [(p, 0) for p in _fake]
print(f"ArtiFact eval fixo: {len(_real)} real + {len(_fake)} fake = {len(ARTIFACT_ITEMS)} imagens")

def sample_subset(dataset, fraction, seed):
    n = int(len(dataset) * fraction)
    idx = random.Random(seed).sample(range(len(dataset)), n)
    return Subset(dataset, idx)

def build_loaders(method, value, seed):
    train_tf = make_transforms(method, value, train=True)
    eval_tf  = make_transforms(method, value, train=False)

    train_ds = sample_subset(datasets.ImageFolder(RAW_DIR / "train", transform=train_tf), SAMPLE_FRACTION, seed)
    valid_ds = sample_subset(datasets.ImageFolder(RAW_DIR / "valid", transform=eval_tf),  SAMPLE_FRACTION, seed)
    # ArtiFact: mesmo conjunto fixo, mas com a preprocess do run aplicada
    artifact_ds = PathListDataset(ARTIFACT_ITEMS, eval_tf)

    train_loader    = DataLoader(train_ds,    batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
    valid_loader    = DataLoader(valid_ds,    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    artifact_loader = DataLoader(artifact_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    return train_loader, valid_loader, artifact_loader

print("Loaders definidos.")

## 3. Treino + Avaliação

In [ ]:
def build_model(seed):
    torch.manual_seed(seed)
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    in_f = model.fc.in_features
    model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(in_f, 2))
    return model.to(DEVICE)

@torch.no_grad()
def evaluate(model, loader):
    """Retorna (auc, acc). AUC usa P(classe=1=real); acc no threshold 0.5."""
    model.eval()
    y_true, y_prob = [], []
    for images, labels in loader:
        images = images.to(DEVICE)
        prob = torch.softmax(model(images), dim=1)[:, 1].cpu().numpy()
        y_prob.extend(prob)
        y_true.extend(labels.numpy())
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    auc = roc_auc_score(y_true, y_prob)
    acc = accuracy_score(y_true, (y_prob >= 0.5).astype(int))
    return float(auc), float(acc)

def run_one(method, value, seed):
    train_loader, valid_loader, artifact_loader = build_loaders(method, value, seed)
    model = build_model(seed)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    for epoch in range(NUM_EPOCHS):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()

    val_auc, val_acc           = evaluate(model, valid_loader)     # mesmo gerador (StyleGAN)
    artifact_auc, artifact_acc = evaluate(model, artifact_loader)  # cross-generator
    del model
    torch.cuda.empty_cache()
    return val_auc, val_acc, artifact_auc, artifact_acc

print("Funções de treino definidas.")

## 4. Loop do Sweep

Salva incrementalmente — se interromper, retoma de onde parou.

In [ ]:
# carrega resultados parciais se existirem
if RESULTS_PATH.exists():
    results = json.loads(RESULTS_PATH.read_text())
    done = {(r["method"], r["value"], r["seed"]) for r in results}
    print(f"Retomando: {len(results)} runs já feitos.")
else:
    results = []
    done = set()

# monta a lista de jobs
jobs = []
for method, values in VALUE_GRID.items():
    for value in values:
        for s in range(N_SEEDS[method]):
            seed = SEEDS[s]
            if (method, value, seed) not in done:
                jobs.append((method, value, seed))

print(f"Jobs restantes: {len(jobs)}")

for method, value, seed in tqdm(jobs, desc="Sweep"):
    start = time.time()
    val_auc, val_acc, artifact_auc, artifact_acc = run_one(method, value, seed)
    elapsed = time.time() - start
    results.append({
        "method": method, "value": value, "seed": seed,
        "val_auc_140k": round(val_auc, 4),
        "val_acc_140k": round(val_acc, 4),
        "artifact_auc": round(artifact_auc, 4),
        "artifact_acc": round(artifact_acc, 4),
        "time": round(elapsed, 1),
    })
    RESULTS_PATH.write_text(json.dumps(results, indent=2))
    # barra widget fica fixa em cima; cada job rola embaixo via tqdm.write
    tqdm.write(f"{method:>10} v={str(value):<5} seed={seed:<4} | "
               f"140k AUC={val_auc:.3f} | artifact AUC={artifact_auc:.3f} | {elapsed:.0f}s")

print("Sweep concluído.")

## 5. Curvas de Resposta (AUC ArtiFact × parâmetro)

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
METRIC = "artifact_auc"

methods = list(VALUE_GRID.keys())
fig, axes = plt.subplots(1, len(methods), figsize=(4 * len(methods), 4), sharey=True)

for ax, method in zip(axes, methods):
    sub = df[df["method"] == method]
    grouped = sub.groupby("value")[METRIC].agg(["mean", "std"]).reset_index()
    ax.errorbar(grouped["value"], grouped["mean"], yerr=grouped["std"].fillna(0),
                marker="o", capsize=3, color="steelblue")
    best = grouped.loc[grouped["mean"].idxmax()]
    ax.scatter([best["value"]], [best["mean"]], color="tomato", s=80, zorder=5,
               label=f"melhor: {best['value']} ({best['mean']:.3f})")
    ax.set_title(method)
    ax.set_xlabel("parâmetro")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

axes[0].set_ylabel("AUC ArtiFact (cross-generator)")
plt.suptitle("Sweep de Pré-processamento — Generalização Cross-Generator", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "sweep_response_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Melhor Configuração por Método

In [ ]:
summary = []
for method in methods:
    sub = df[df["method"] == method]
    grouped = sub.groupby("value")[METRIC].agg(["mean", "std"]).reset_index()
    best = grouped.loc[grouped["mean"].idxmax()]
    summary.append({
        "method": method,
        "best_value": best["value"],
        "artifact_auc": round(best["mean"], 4),
        "std": round(best["std"], 4) if not np.isnan(best["std"]) else 0.0,
    })

summary_df = pd.DataFrame(summary).sort_values("artifact_auc", ascending=False)
print(summary_df.to_string(index=False))

best_overall = summary_df.iloc[0]
print(f"\n→ Melhor método: {best_overall['method']} "
      f"(valor={best_overall['best_value']}, AUC ArtiFact={best_overall['artifact_auc']:.4f})")

summary_df.to_json(RESULTS_DIR / "sweep_summary.json", orient="records", indent=2)
print("Resumo salvo em:", RESULTS_DIR / "sweep_summary.json")